### Important Prerequisites

Before running this notebook, ensure the SSE server is already running.
You can start it in a separate terminal using:
```bash
uv run server.py
```
Make sure it is listening on port 8050.

In [3]:
import asyncio
import nest_asyncio
from mcp import ClientSession
from mcp.client.sse import sse_client
from mcp.types import TextContent

nest_asyncio.apply()  # Needed to run interactive python
print("Imports loaded. nest_asyncio applied — async code can now run in Jupyter.")

Imports loaded. nest_asyncio applied — async code can now run in Jupyter.


In [4]:
server_url = "http://localhost:8050/sse"

async def explore_tools():
    # Connect to the server using SSE
    print(f"Connecting to server at {server_url}...")
    async with sse_client(server_url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            print("ClientSession created — MCP protocol handshake in progress...")
            
            # Initialize the connection
            await session.initialize()
            print("Session initialized! Client and server have negotiated capabilities.\n")

            # List available tools
            tools_result = await session.list_tools()
            print("Available tools advertised by the server:")
            for tool in tools_result.tools:
                print(f"   - {tool.name}: {tool.description}")

asyncio.run(explore_tools())


Connecting to server at http://localhost:8050/sse...
ClientSession created — MCP protocol handshake in progress...
Session initialized! Client and server have negotiated capabilities.

Available tools advertised by the server:
   - add: Add two numbers together


In [5]:
async def call_add_tool():
    print(f"Connecting to server at {server_url}...")
    async with sse_client(server_url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            print("Calling tool: 'add' with arguments a=2, b=3")

            # Call our calculator tool
            result = await session.call_tool("add", arguments={"a": 2, "b": 3})
            content = result.content[0]
            
            if isinstance(content, TextContent):
                print(f"Raw response content type : TextContent")
                print(f"Result: 2 + 3 = {content.text}")

asyncio.run(call_add_tool())


Connecting to server at http://localhost:8050/sse...
Calling tool: 'add' with arguments a=2, b=3
Raw response content type : TextContent
Result: 2 + 3 = 5
